# Cross-Attention and VLA Architectures

**Lecture 3, Part 2 — Vizuara Modern Robot Learning Bootcamp**

We explore how VLAs turn vision-language understanding into robot actions, building every component from scratch:

| Component | What it does | Key Insight |
|-----------|-------------|-------------|
| Self-Attention (recap) | Tokens attend to each other | Square N×N attention matrix |
| Action Tokenization | Continuous actions → discrete bins | OpenVLA’s 256-bin quantile encoding |
| Autoregressive Bottleneck | Generate actions one token at a time | 7B model × 112 tokens = slow |
| Cross-Attention | Actions query observations | Rectangular M×N attention matrix |
| Conditioning | Observation tokens steer action generation | output_i = Σ_j α_ij · V_j^obs |
| OpenVLA Pipeline | Dual encoder + MLP projection + autoregressive | 7B parameters |
| SmolVLA Action Expert | Interleaved cross-attn + self-attn blocks | 100M parameters |
| Attention Masks | Causal, block-wise, interleaved patterns | What each architecture can see |

**Running example:** A robot workspace with a red cup, blue plate, and green cube. The instruction is **"pick up the red cup"**.

---

In [ ]:
!pip install -q torch numpy matplotlib 2>&1 | tail -3

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
import math
import time

# Vizuara color palette
ACCENT = '#D97757'
BLUE   = '#5B8CB8'
TEAL   = '#7DA488'
PURPLE = '#9B7EC8'
WARM   = '#C4956A'
BG     = '#FAF8F5'

plt.rcParams['figure.facecolor'] = BG
plt.rcParams['axes.facecolor'] = BG
plt.rcParams['font.size'] = 12

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

---
## Step 1: Self-Attention Recap

Before introducing cross-attention, let’s nail down self-attention in its simplest form.

The core formula (8 lines of code):

1. Project input into Q, K, V
2. Compute scores = Q · K^T / √d_k
3. Apply softmax to get weights
4. Output = weights · V

The attention matrix is **square** (N×N) because every token attends to every other token from the **same** sequence.

In [ ]:
class SelfAttention(nn.Module):
    """Scaled dot-product self-attention in ~8 lines of forward()."""
    def __init__(self, d_model=64, n_heads=4):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        # x: [seq_len, d_model]
        Q = self.W_q(x)                                  # [seq_len, d_model]
        K = self.W_k(x)                                  # [seq_len, d_model]
        V = self.W_v(x)                                  # [seq_len, d_model]
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_model)  # [seq_len, seq_len]
        weights = F.softmax(scores, dim=-1)              # each row sums to 1
        output = weights @ V                             # [seq_len, d_model]
        output = self.W_o(output)                        # project back
        return output, weights


# Test: 10 action tokens of dimension 64
D_MODEL = 64
N_HEADS = 4
N_ACTION_TOKENS = 10

self_attn = SelfAttention(d_model=D_MODEL, n_heads=N_HEADS)
action_tokens = torch.randn(N_ACTION_TOKENS, D_MODEL)

with torch.no_grad():
    sa_out, sa_weights = self_attn(action_tokens)

print(f"Self-Attention:")
print(f"  Input:            {list(action_tokens.shape)}  — {N_ACTION_TOKENS} action tokens × {D_MODEL}-d")
print(f"  Output:           {list(sa_out.shape)}  — same shape (each token enriched)")
print(f"  Attention matrix: {list(sa_weights.shape)}  — SQUARE: every token attends to every token")
print(f"  Parameters:       {sum(p.numel() for p in self_attn.parameters()):,}")

In [ ]:
# Visualize the square attention matrix
fig, ax = plt.subplots(figsize=(8, 7))
w = sa_weights.detach().numpy()
im = ax.imshow(w, cmap='YlOrRd', vmin=0)
ax.set_xticks(range(N_ACTION_TOKENS))
ax.set_yticks(range(N_ACTION_TOKENS))
ax.set_xticklabels([f'a{i}' for i in range(N_ACTION_TOKENS)], fontsize=10)
ax.set_yticklabels([f'a{i}' for i in range(N_ACTION_TOKENS)], fontsize=10)
ax.set_xlabel('Keys (attends TO)', fontsize=12, fontweight='bold')
ax.set_ylabel('Queries (attends FROM)', fontsize=12, fontweight='bold')

for i in range(N_ACTION_TOKENS):
    for j in range(N_ACTION_TOKENS):
        val = w[i, j]
        color = 'white' if val > 0.15 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8, color=color)

plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Self-Attention: Square Matrix (10×10)\nEvery action token attends to every other',
             fontsize=13, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key observation: The matrix is SQUARE because Q, K, V all come from the SAME sequence.")
print("Each row sums to 1.0 (softmax). Each row = one token's attention distribution.")

---
## Step 2: Action Tokenization (OpenVLA)

OpenVLA converts **continuous** robot actions into **discrete** tokens using **256-bin quantile tokenization**.

Why? Because the LLM backbone (Llama) is a language model — it outputs tokens from a vocabulary, not continuous numbers. So we need to:

1. **Collect training data** → compute quantile bin edges for each action dimension
2. **Quantize** continuous value → which bin does it fall into?
3. **Dequantize** bin ID → bin center value (for execution)

This is conceptually similar to color quantization in images (16M colors → 256 palette entries).

In [ ]:
# Action tokenization: 256-bin quantile encoding
N_BINS = 256
N_ACTION_DIMS = 7   # 6-DOF joint positions + 1 gripper
N_SAMPLES = 5000    # training dataset size

torch.manual_seed(42)

# Step 1: Generate synthetic continuous actions (from "training data")
# Each dim has a different distribution to be realistic
training_actions = torch.zeros(N_SAMPLES, N_ACTION_DIMS)
dim_names = ['joint_1', 'joint_2', 'joint_3', 'joint_4', 'joint_5', 'joint_6', 'gripper']

for d in range(N_ACTION_DIMS - 1):  # joints: mixture of modes
    center = (d - 3) * 0.3
    training_actions[:, d] = torch.randn(N_SAMPLES) * 0.5 + center

# Gripper: bimodal (open or closed)
gripper_open = torch.randn(N_SAMPLES // 2) * 0.1 + 0.9
gripper_closed = torch.randn(N_SAMPLES // 2 + N_SAMPLES % 2) * 0.1 + 0.1
training_actions[:, 6] = torch.cat([gripper_open, gripper_closed])

print(f"Training actions shape: {list(training_actions.shape)}")
print(f"  {N_SAMPLES} samples × {N_ACTION_DIMS} dims ({', '.join(dim_names)})")
print(f"\nValue ranges per dim:")
for d in range(N_ACTION_DIMS):
    lo, hi = training_actions[:, d].min().item(), training_actions[:, d].max().item()
    print(f"  {dim_names[d]:>8s}: [{lo:+.3f}, {hi:+.3f}]")

In [ ]:
# Step 2: Compute quantile bin edges from training statistics
quantiles = torch.linspace(0, 1, N_BINS + 1)  # 257 edges for 256 bins
bin_edges = torch.zeros(N_ACTION_DIMS, N_BINS + 1)

for d in range(N_ACTION_DIMS):
    bin_edges[d] = torch.quantile(training_actions[:, d].float(), quantiles)

# Compute bin centers (midpoint of each bin)
bin_centers = (bin_edges[:, :-1] + bin_edges[:, 1:]) / 2  # [N_ACTION_DIMS, N_BINS]

print(f"Bin edges shape:   {list(bin_edges.shape)}   — {N_BINS + 1} edges per dim")
print(f"Bin centers shape: {list(bin_centers.shape)} — {N_BINS} centers per dim")
print(f"\nExample: joint_1 first 5 bin edges: {bin_edges[0, :5].tolist()}")
print(f"Example: joint_1 first 5 bin centers: {bin_centers[0, :5].tolist()}")

In [ ]:
def quantize(continuous_actions, bin_edges):
    """Continuous values -> discrete bin IDs."""
    n_dims = continuous_actions.shape[1]
    bin_ids = torch.zeros_like(continuous_actions, dtype=torch.long)
    for d in range(n_dims):
        # torch.bucketize: find which bin each value falls into
        bin_ids[:, d] = torch.bucketize(continuous_actions[:, d], bin_edges[d, 1:-1])
    return bin_ids


def dequantize(bin_ids, bin_centers):
    """Discrete bin IDs -> continuous values (bin center)."""
    n_dims = bin_ids.shape[1]
    continuous = torch.zeros(bin_ids.shape, dtype=torch.float32)
    for d in range(n_dims):
        continuous[:, d] = bin_centers[d][bin_ids[:, d]]
    return continuous


# Test: quantize then dequantize
N_TEST = 20
test_actions = training_actions[:N_TEST]
test_bin_ids = quantize(test_actions, bin_edges)
test_reconstructed = dequantize(test_bin_ids, bin_centers)
recon_error = (test_actions - test_reconstructed).abs()

print(f"Original actions:      {list(test_actions.shape)}")
print(f"Bin IDs:               {list(test_bin_ids.shape)} (values in [0, {N_BINS - 1}])")
print(f"Reconstructed actions: {list(test_reconstructed.shape)}")
print(f"\nMean absolute error:   {recon_error.mean().item():.6f}")
print(f"Max absolute error:    {recon_error.max().item():.6f}")
print(f"\nSample bin IDs (first 3 samples): {test_bin_ids[:3].tolist()}")

In [ ]:
# Visualize: continuous vs reconstructed actions + reconstruction error heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: continuous vs reconstructed (line plot per dim)
ax = axes[0]
dim_colors = [ACCENT, BLUE, TEAL, PURPLE, WARM, '#666666', '#333333']
for d in range(N_ACTION_DIMS):
    ax.plot(range(N_TEST), test_actions[:, d].numpy(), 'o-',
            color=dim_colors[d], alpha=0.7, markersize=5, label=f'{dim_names[d]} (orig)')
    ax.plot(range(N_TEST), test_reconstructed[:, d].numpy(), 'x--',
            color=dim_colors[d], alpha=0.4, markersize=5)
ax.set_xlabel('Sample index', fontsize=11)
ax.set_ylabel('Action value', fontsize=11)
ax.set_title(f'Continuous vs Reconstructed ({N_BINS} bins)',
             fontsize=13, color=ACCENT, fontweight='bold')
ax.legend(fontsize=8, ncol=2, loc='upper right')
ax.grid(True, alpha=0.3)

# Right: reconstruction error heatmap (7 dims x N samples)
ax = axes[1]
err_data = recon_error.numpy().T  # [N_ACTION_DIMS, N_TEST]
im = ax.imshow(err_data, aspect='auto', cmap='YlOrRd', vmin=0)
ax.set_yticks(range(N_ACTION_DIMS))
ax.set_yticklabels(dim_names, fontsize=10)
ax.set_xlabel('Sample index', fontsize=11)
ax.set_title(f'Reconstruction Error per Dim \u00d7 Sample',
             fontsize=13, color=ACCENT, fontweight='bold')
for i in range(N_ACTION_DIMS):
    for j in range(N_TEST):
        val = err_data[i, j]
        color = 'white' if val > err_data.max() * 0.5 else 'black'
        ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=6, color=color)
plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

print(f"With {N_BINS} bins, quantization error is very small (mean {recon_error.mean():.6f}).")
print(f"The 'x' markers (reconstructed) overlap almost perfectly with 'o' (original).")
print(f"\nKey trade-off: More bins = better precision, but larger vocabulary for the LLM.")
print(f"OpenVLA uses 256 bins — a good balance between precision and vocab size.")

---
## Step 3: The Autoregressive Bottleneck

OpenVLA generates action tokens **one at a time** (autoregressive). Each token requires a full forward pass through the 7B model.

For a single timestep: 7 dimensions × 1 forward pass each = 7 sequential passes.

For action chunking (16 timesteps): 16 × 7 = **112 sequential passes**.

This is the fundamental speed bottleneck that SmolVLA solves with **parallel flow matching**.

In [ ]:
# Simulate sequential generation timing
FORWARD_PASS_7B = 50   # ms per token for 7B model (OpenVLA)
FORWARD_PASS_100M = 30 # ms per token for 100M model (SmolVLA action expert)

# OpenVLA: autoregressive, one token at a time
openvla_single = 7 * FORWARD_PASS_7B      # 7 dims, 1 timestep
openvla_chunk = 112 * FORWARD_PASS_7B     # 7 dims x 16 timesteps

# SmolVLA: parallel flow matching — all action dims generated in one pass
# 10 denoising steps, each is one forward pass through 100M expert
smolvla_single = 10 * FORWARD_PASS_100M   # 10 denoising steps, 1 timestep
smolvla_chunk = 10 * FORWARD_PASS_100M    # 10 denoising steps, 16 timesteps (parallel!)

print("=" * 60)
print("INFERENCE SPEED COMPARISON")
print("=" * 60)
print(f"\nOpenVLA (7B, autoregressive):")
print(f"  Single step (7 tokens):    {7} × {FORWARD_PASS_7B}ms = {openvla_single:>6}ms ({openvla_single/1000:.2f}s)")
print(f"  16-step chunk (112 tokens): {112} × {FORWARD_PASS_7B}ms = {openvla_chunk:>6}ms ({openvla_chunk/1000:.1f}s)")
print(f"\nSmolVLA (100M action expert, flow matching):")
print(f"  Single step (parallel):    {10} × {FORWARD_PASS_100M}ms = {smolvla_single:>6}ms ({smolvla_single/1000:.2f}s)")
print(f"  16-step chunk (parallel):  {10} × {FORWARD_PASS_100M}ms = {smolvla_chunk:>6}ms ({smolvla_chunk/1000:.2f}s)")
print(f"\nSpeedup (chunk): {openvla_chunk / smolvla_chunk:.1f}x faster with SmolVLA")
print(f"\nKey insight: SmolVLA's chunk time is THE SAME as single-step time!")
print(f"Flow matching generates all dims + all timesteps in parallel.")

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(12, 6))

methods = ['OpenVLA\n(1 step)', 'OpenVLA\n(16-step chunk)', 'SmolVLA\n(1 step)', 'SmolVLA\n(16-step chunk)']
times = [openvla_single, openvla_chunk, smolvla_single, smolvla_chunk]
colors = [ACCENT, ACCENT, TEAL, TEAL]
hatches = ['', '//', '', '//']

bars = ax.bar(methods, times, color=colors, edgecolor='white', linewidth=2)
for bar, h in zip(bars, hatches):
    bar.set_hatch(h)

for i, (t, method) in enumerate(zip(times, methods)):
    ax.text(i, t + 80, f'{t}ms\n({t/1000:.1f}s)', ha='center', fontsize=11, fontweight='bold')

ax.set_ylabel('Inference Time (ms)', fontsize=12, fontweight='bold')
ax.set_title('Autoregressive (OpenVLA) vs Parallel Flow Matching (SmolVLA)',
             fontsize=14, color=ACCENT, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Add annotation
ax.annotate(f'{openvla_chunk/smolvla_chunk:.0f}x\nfaster!',
            xy=(3, smolvla_chunk), xytext=(2.5, openvla_chunk * 0.6),
            fontsize=14, color=TEAL, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=TEAL, lw=2))

plt.tight_layout()
plt.show()

print("The autoregressive bottleneck scales LINEARLY with chunk size.")
print("Flow matching is constant-time regardless of chunk size — all tokens generated in parallel.")
print("\nThis is why SmolVLA can run on a Raspberry Pi while OpenVLA needs an A100.")

---
## Step 4: From Self-Attention to Cross-Attention

The **only** difference between self-attention and cross-attention:

| | Self-Attention | Cross-Attention |
|---|---|---|
| Q comes from | same sequence | **action** tokens |
| K comes from | same sequence | **observation** tokens |
| V comes from | same sequence | **observation** tokens |
| Matrix shape | N×N (square) | M×N (**rectangular**) |

Cross-attention lets actions **query** observations: "What in the visual scene is relevant to my next move?"

In [ ]:
class CrossAttention(nn.Module):
    """Cross-attention: Q from actions, K/V from observations.

    The ONLY difference from self-attention: K, V come from a different source.
    """
    def __init__(self, d_action=64, d_obs=64, n_heads=4):
        super().__init__()
        self.d_action = d_action
        self.n_heads = n_heads
        self.d_head = d_action // n_heads
        self.W_q = nn.Linear(d_action, d_action, bias=False)   # Q from actions
        self.W_k = nn.Linear(d_obs, d_action, bias=False)      # K from observations
        self.W_v = nn.Linear(d_obs, d_action, bias=False)      # V from observations
        self.W_o = nn.Linear(d_action, d_action, bias=False)

    def forward(self, action_tokens, obs_tokens):
        # action_tokens: [M, d_action]  — queries
        # obs_tokens:    [N, d_obs]     — keys/values
        Q = self.W_q(action_tokens)                           # [M, d_action]
        K = self.W_k(obs_tokens)                              # [N, d_action]
        V = self.W_v(obs_tokens)                              # [N, d_action]
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_action)  # [M, N]
        weights = F.softmax(scores, dim=-1)                   # each row sums to 1
        output = weights @ V                                  # [M, d_action]
        output = self.W_o(output)
        return output, weights


# Test: 50 action tokens querying 64 observation tokens
N_ACT = 50
N_OBS = 64

cross_attn = CrossAttention(d_action=D_MODEL, d_obs=D_MODEL, n_heads=N_HEADS)
act_tokens = torch.randn(N_ACT, D_MODEL)
obs_tokens = torch.randn(N_OBS, D_MODEL)

with torch.no_grad():
    ca_out, ca_weights = cross_attn(act_tokens, obs_tokens)

print(f"Cross-Attention:")
print(f"  Action tokens (Q): {list(act_tokens.shape)}  — {N_ACT} queries")
print(f"  Obs tokens (K,V):  {list(obs_tokens.shape)}  — {N_OBS} keys/values")
print(f"  Output:            {list(ca_out.shape)}  — same shape as action tokens")
print(f"  Attention matrix:  {list(ca_weights.shape)}  — RECTANGULAR: {N_ACT}×{N_OBS}")
print(f"  Parameters:        {sum(p.numel() for p in cross_attn.parameters()):,}")

In [ ]:
# Side-by-side comparison: self-attention (square) vs cross-attention (rectangular)
sa_test = SelfAttention(d_model=D_MODEL, n_heads=N_HEADS)
sa_input = torch.randn(N_ACT, D_MODEL)

with torch.no_grad():
    _, sa_w = sa_test(sa_input)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Left: Self-attention (50x50 square)
ax = axes[0]
im0 = ax.imshow(sa_w.detach().numpy(), cmap='YlOrRd', vmin=0, aspect='auto')
ax.set_xlabel('Keys (actions)', fontsize=12, fontweight='bold')
ax.set_ylabel('Queries (actions)', fontsize=12, fontweight='bold')
ax.set_title(f'Self-Attention ({N_ACT}×{N_ACT})\nSquare: Q, K, V from SAME source',
             fontsize=13, color=ACCENT, fontweight='bold')
plt.colorbar(im0, ax=ax, shrink=0.8)

# Right: Cross-attention (50x64 rectangular)
ax = axes[1]
im1 = ax.imshow(ca_weights.detach().numpy(), cmap='YlOrRd', vmin=0, aspect='auto')
ax.set_xlabel('Keys (observations)', fontsize=12, fontweight='bold', color=BLUE)
ax.set_ylabel('Queries (actions)', fontsize=12, fontweight='bold', color=ACCENT)
ax.set_title(f'Cross-Attention ({N_ACT}×{N_OBS})\nRectangular: Q from actions, K/V from observations',
             fontsize=13, color=TEAL, fontweight='bold')
plt.colorbar(im1, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

print("Left (self-attention): 50×50 — every action token attends to every other action token.")
print("Right (cross-attention): 50×64 — every action token queries every observation token.")
print("\nThe rectangular shape is THE defining feature of cross-attention.")
print("It means: actions can READ FROM observations, but observations are unchanged.")

---
## Step 5: Cross-Attention = Conditioning

Cross-attention is the mechanism by which **observations steer action generation**.

Mathematically: $\text{output}_i = \sum_j \alpha_{ij} \cdot V_j^{\text{obs}}$

Each action token $i$ computes attention weights $\alpha_{ij}$ over all observation tokens $j$, then takes a weighted sum of their values.

Let's make this concrete: in our "pick up the red cup" scenario, **reaching actions** should attend to the cup's position, while **placing actions** should attend to the target location.

In [ ]:
# Concrete demo: 50 action tokens querying 64 VLM observation tokens
# We'll engineer the scenario so the attention pattern is interpretable

torch.manual_seed(42)

N_ACT_DEMO = 50   # action sequence
N_OBS_DEMO = 64   # VLM output tokens
D = 64

# Create observation tokens with semantic structure
# Tokens 0-20: background scene features
# Tokens 21-30: "red cup" region (position ~3 in workspace)
# Tokens 31-40: "blue plate" region
# Tokens 50-60: "bowl at position 60" region (target)

obs_demo = torch.randn(N_OBS_DEMO, D) * 0.3  # low-energy background

# Make "cup" tokens (21-30) have a strong, distinctive signal
cup_signal = torch.randn(1, D) * 2.0
obs_demo[21:31] = cup_signal + torch.randn(10, D) * 0.1

# Make "bowl/target" tokens (50-60) have a different strong signal
bowl_signal = torch.randn(1, D) * 2.0
obs_demo[50:61] = bowl_signal + torch.randn(11, D) * 0.1

# Create action tokens with semantic structure
# Tokens 0-19: reaching phase (should attend to cup)
# Tokens 20-49: placing phase (should attend to bowl/target)

act_demo = torch.randn(N_ACT_DEMO, D) * 0.3

# Make reaching tokens similar to cup signal (so Q*K is high for cup)
act_demo[:20] = cup_signal * 0.5 + torch.randn(20, D) * 0.2

# Make placing tokens similar to bowl signal
act_demo[20:] = bowl_signal * 0.5 + torch.randn(30, D) * 0.2

# Build cross-attention with identity-like weights to preserve the signal
ca_demo = CrossAttention(d_action=D, d_obs=D, n_heads=N_HEADS)

with torch.no_grad():
    ca_demo_out, ca_demo_weights = ca_demo(act_demo, obs_demo)

print(f"Scenario: 'pick up the red cup and place it in the bowl'")
print(f"  Action tokens 0-19:  reaching phase (should look at cup)")
print(f"  Action tokens 20-49: placing phase (should look at bowl)")
print(f"  Obs tokens 21-30:    'red cup' features")
print(f"  Obs tokens 50-60:    'bowl/target' features")
print(f"\nAttention weights shape: {list(ca_demo_weights.shape)}")

In [ ]:
# Visualize cross-attention weights showing reaching->cup, placing->bowl pattern
fig, ax = plt.subplots(figsize=(16, 10))

w = ca_demo_weights.detach().numpy()
im = ax.imshow(w, cmap='YlOrRd', aspect='auto', vmin=0)
ax.set_xlabel('Observation tokens (VLM output)', fontsize=13, fontweight='bold')
ax.set_ylabel('Action tokens', fontsize=13, fontweight='bold')
ax.set_title('Cross-Attention: Actions Querying Observations\n"pick up the red cup and place it in the bowl"',
             fontsize=14, color=ACCENT, fontweight='bold')

# Mark regions
ax.add_patch(Rectangle((20.5, -0.5), 10, 20, fill=False,
                        edgecolor=ACCENT, linewidth=3, linestyle='--'))
ax.text(25.5, 9.5, 'reaching\n\u2192 cup', ha='center', va='center',
        fontsize=11, color=ACCENT, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax.add_patch(Rectangle((49.5, 19.5), 11, 30, fill=False,
                        edgecolor=BLUE, linewidth=3, linestyle='--'))
ax.text(55, 34.5, 'placing\n\u2192 bowl', ha='center', va='center',
        fontsize=11, color=BLUE, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Add phase labels on y-axis
ax.annotate('REACH phase', xy=(-0.02, 0.85), xycoords='axes fraction',
            fontsize=11, color=ACCENT, fontweight='bold', rotation=90, va='center')
ax.annotate('PLACE phase', xy=(-0.02, 0.4), xycoords='axes fraction',
            fontsize=11, color=BLUE, fontweight='bold', rotation=90, va='center')

# Mark observation regions on x-axis
ax.axvline(x=20.5, color='gray', linestyle=':', alpha=0.5)
ax.axvline(x=30.5, color='gray', linestyle=':', alpha=0.5)
ax.axvline(x=49.5, color='gray', linestyle=':', alpha=0.5)
ax.axvline(x=60.5, color='gray', linestyle=':', alpha=0.5)
ax.text(10, N_ACT_DEMO + 2.5, 'background', ha='center', fontsize=9, color='gray')
ax.text(25.5, N_ACT_DEMO + 2.5, 'cup', ha='center', fontsize=10, color=ACCENT, fontweight='bold')
ax.text(40, N_ACT_DEMO + 2.5, 'other', ha='center', fontsize=9, color='gray')
ax.text(55, N_ACT_DEMO + 2.5, 'bowl', ha='center', fontsize=10, color=BLUE, fontweight='bold')

plt.colorbar(im, ax=ax, shrink=0.7, label='Attention weight')
plt.tight_layout()
plt.show()

# Print attention statistics
reach_to_cup = w[:20, 21:31].mean()
reach_to_bowl = w[:20, 50:61].mean()
place_to_cup = w[20:, 21:31].mean()
place_to_bowl = w[20:, 50:61].mean()

print(f"Mean attention weights:")
print(f"  Reaching \u2192 cup region:  {reach_to_cup:.4f}")
print(f"  Reaching \u2192 bowl region: {reach_to_bowl:.4f}")
print(f"  Placing  \u2192 cup region:  {place_to_cup:.4f}")
print(f"  Placing  \u2192 bowl region: {place_to_bowl:.4f}")
print(f"\nCross-attention is the CONDITIONING mechanism:")
print(f"  output_i = sum_j alpha_ij * V_j^obs")
print(f"  Each action token becomes a weighted mix of observation features.")

---
## Step 6: OpenVLA Architecture Simulation

OpenVLA’s pipeline:

1. **Dual visual encoder**: SigLIP (semantic) + DinoV2 (spatial) → concatenated visual features
2. **MLP projection**: Project concatenated features to Llama’s hidden dimension
3. **Autoregressive decoder**: Llama backbone generates 7 action tokens one at a time

Total: ~7B parameters. Let’s simulate the full pipeline with correct tensor shapes.

In [ ]:
class OpenVLAPipeline(nn.Module):
    """Simplified OpenVLA pipeline (shape simulation).

    Real OpenVLA uses pre-trained SigLIP, DinoV2, and Llama-2-7B.
    We simulate shapes and parameter counts.
    """
    def __init__(self):
        super().__init__()

        # 1. Dual Visual Encoder (simulated)
        self.siglip_dim = 1152    # SigLIP-SO400M output dim
        self.dinov2_dim = 1024    # DinoV2-Large output dim
        self.n_patches = 256     # 16x16 patches from 224x224 image
        self.concat_dim = self.siglip_dim + self.dinov2_dim  # 2176

        # 2. MLP Projection (2 layers) -> Llama hidden dim
        self.llama_dim = 4096    # Llama-2-7B hidden size
        self.projector = nn.Sequential(
            nn.Linear(self.concat_dim, self.llama_dim),
            nn.GELU(),
            nn.Linear(self.llama_dim, self.llama_dim),
        )

        # 3. Action head (simulated: just tracks the autoregressive shape)
        self.action_vocab_size = 256  # 256 bins
        self.n_action_dims = 7
        self.action_head = nn.Linear(self.llama_dim, self.action_vocab_size)

    def forward(self, image):
        """Simulate forward pass, printing shapes at each stage."""
        batch = image.shape[0]

        # Stage 1: Dual encoder
        siglip_out = torch.randn(batch, self.n_patches, self.siglip_dim)
        dinov2_out = torch.randn(batch, self.n_patches, self.dinov2_dim)
        print(f"Stage 1: Dual Visual Encoder")
        print(f"  SigLIP output:  {list(siglip_out.shape)}  ({self.n_patches} patches \u00d7 {self.siglip_dim}-d)")
        print(f"  DinoV2 output:  {list(dinov2_out.shape)}  ({self.n_patches} patches \u00d7 {self.dinov2_dim}-d)")

        # Stage 2: Channel concatenation
        concat = torch.cat([siglip_out, dinov2_out], dim=-1)
        print(f"\nStage 2: Channel Concatenation")
        print(f"  Concatenated:   {list(concat.shape)}  ({self.siglip_dim} + {self.dinov2_dim} = {self.concat_dim})")

        # Stage 3: MLP Projection
        projected = self.projector(concat)
        print(f"\nStage 3: MLP Projection (2-layer)")
        print(f"  Projected:      {list(projected.shape)}  (\u2192 Llama hidden dim {self.llama_dim})")

        # Stage 4: Autoregressive decoder (shape tracking)
        # In reality, Llama processes [vision_tokens + text_tokens + action_tokens]
        # and generates action tokens one at a time
        text_tokens_sim = torch.randn(batch, 20, self.llama_dim)  # ~20 text tokens
        full_sequence = torch.cat([projected, text_tokens_sim], dim=1)
        print(f"\nStage 4: Autoregressive Decoder (Llama)")
        print(f"  Input sequence: {list(full_sequence.shape)}")
        print(f"    = {self.n_patches} vision + 20 text = {self.n_patches + 20} context tokens")

        # Generate 7 action tokens autoregressively
        action_logits = self.action_head(full_sequence[:, -1:, :])  # last token -> next action
        print(f"  Action logits:  {list(action_logits.shape)}  (per token: {self.action_vocab_size} bins)")
        print(f"  Full generation: 7 sequential forward passes for 7-dim action")

        return action_logits


# Build and run
openvla = OpenVLAPipeline()
dummy_image = torch.randn(1, 3, 224, 224)

print("=" * 60)
print("OpenVLA ARCHITECTURE SIMULATION")
print("=" * 60)
print()

with torch.no_grad():
    _ = openvla(dummy_image)

In [ ]:
# Parameter counts for each component
proj_params = sum(p.numel() for p in openvla.projector.parameters())
head_params = sum(p.numel() for p in openvla.action_head.parameters())

# Simulated counts for the full model (we don't instantiate 7B params)
siglip_params = 400_000_000    # SigLIP-SO400M
dinov2_params = 304_000_000    # DinoV2-Large
llama_params = 6_700_000_000   # Llama-2-7B

print("\nParameter Count Breakdown:")
print(f"  SigLIP encoder:    {siglip_params:>14,}  (400M, frozen during fine-tuning)")
print(f"  DinoV2 encoder:    {dinov2_params:>14,}  (304M, frozen during fine-tuning)")
print(f"  MLP projector:     {proj_params:>14,}  (trainable)")
print(f"  Llama-2 backbone:  {llama_params:>14,}  (6.7B, LoRA fine-tuned)")
print(f"  Action head:       {head_params:>14,}  (trainable)")
print(f"  {'':->50}")
total = siglip_params + dinov2_params + proj_params + llama_params + head_params
print(f"  Total:             {total:>14,}  (~{total/1e9:.1f}B)")

---
## Step 7: SmolVLA Action Expert

SmolVLA replaces the monolithic autoregressive decoder with a tiny **action expert** (~100M params) that uses:

1. **Cross-attention**: Action tokens query VLM features ("what does the world look like?")
2. **Self-attention**: Action tokens communicate with each other ("coordinate the trajectory")

These are **interleaved** in each block: CA first (read the world), then SA (coordinate actions).

Information flows: **world \u2192 actions** (CA) then **actions \u2192 actions** (SA).

In [ ]:
class SmolVLAActionExpertBlock(nn.Module):
    """One block of SmolVLA's action expert:
    1. Cross-attention (Q from actions, KV from VLM features)
    2. Self-attention (causal, among action tokens only)
    Each with LayerNorm + FFN.
    """
    def __init__(self, d_action=64, d_obs=64, n_heads=4, d_ff=256):
        super().__init__()
        # Cross-attention sublayer
        self.norm_ca = nn.LayerNorm(d_action)
        self.cross_attn = CrossAttention(d_action=d_action, d_obs=d_obs, n_heads=n_heads)
        self.ffn_ca = nn.Sequential(
            nn.Linear(d_action, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_action),
        )
        self.norm_ffn_ca = nn.LayerNorm(d_action)

        # Self-attention sublayer
        self.norm_sa = nn.LayerNorm(d_action)
        self.self_attn = SelfAttention(d_model=d_action, n_heads=n_heads)
        self.ffn_sa = nn.Sequential(
            nn.Linear(d_action, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_action),
        )
        self.norm_ffn_sa = nn.LayerNorm(d_action)

    def forward(self, action_tokens, obs_tokens):
        # Step 1: Cross-attention (world -> actions)
        normed = self.norm_ca(action_tokens)
        ca_out, ca_weights = self.cross_attn(normed, obs_tokens)
        action_tokens = action_tokens + ca_out           # residual
        action_tokens = action_tokens + self.ffn_ca(self.norm_ffn_ca(action_tokens))  # FFN

        # Step 2: Self-attention (actions -> actions, causal)
        normed = self.norm_sa(action_tokens)
        sa_out, sa_weights = self.self_attn(normed)
        action_tokens = action_tokens + sa_out           # residual
        action_tokens = action_tokens + self.ffn_sa(self.norm_ffn_sa(action_tokens))  # FFN

        return action_tokens, ca_weights, sa_weights


# Test a single block
block = SmolVLAActionExpertBlock(d_action=D_MODEL, d_obs=D_MODEL, n_heads=N_HEADS, d_ff=256)
test_act = torch.randn(N_ACT, D_MODEL)
test_obs = torch.randn(N_OBS, D_MODEL)

with torch.no_grad():
    block_out, block_ca_w, block_sa_w = block(test_act, test_obs)

block_params = sum(p.numel() for p in block.parameters())
print(f"SmolVLA Action Expert Block:")
print(f"  Input actions:  {list(test_act.shape)}")
print(f"  Input obs:      {list(test_obs.shape)}")
print(f"  Output actions: {list(block_out.shape)}")
print(f"  CA weights:     {list(block_ca_w.shape)}  (actions \u2192 observations)")
print(f"  SA weights:     {list(block_sa_w.shape)}  (actions \u2192 actions)")
print(f"  Parameters:     {block_params:,}")

In [ ]:
# Stack 6 blocks (like the real SmolVLA action expert)
N_BLOCKS = 6

class SmolVLAActionExpert(nn.Module):
    """Full SmolVLA action expert: N interleaved CA+SA blocks."""
    def __init__(self, n_blocks=6, d_action=64, d_obs=64, n_heads=4, d_ff=256):
        super().__init__()
        self.blocks = nn.ModuleList([
            SmolVLAActionExpertBlock(d_action, d_obs, n_heads, d_ff)
            for _ in range(n_blocks)
        ])
        self.final_norm = nn.LayerNorm(d_action)

    def forward(self, action_tokens, obs_tokens, verbose=False):
        for i, blk in enumerate(self.blocks):
            action_tokens, ca_w, sa_w = blk(action_tokens, obs_tokens)
            if verbose:
                print(f"  Block {i}: actions {list(action_tokens.shape)} | "
                      f"CA {list(ca_w.shape)} | SA {list(sa_w.shape)}")
        return self.final_norm(action_tokens)


expert = SmolVLAActionExpert(n_blocks=N_BLOCKS, d_action=D_MODEL, d_obs=D_MODEL,
                             n_heads=N_HEADS, d_ff=256)

print(f"SmolVLA Action Expert ({N_BLOCKS} blocks):")
print(f"  Forward pass shape trace:")

with torch.no_grad():
    expert_out = expert(test_act, test_obs, verbose=True)

expert_params = sum(p.numel() for p in expert.parameters())
print(f"\n  Final output:   {list(expert_out.shape)}")
print(f"  Total params:   {expert_params:,}")
print(f"\nInformation flow in each block:")
print(f"  1. Cross-attention: world \u2192 actions  (actions READ observations)")
print(f"  2. Self-attention:  actions \u2192 actions (actions COORDINATE with each other)")
print(f"  After 6 blocks, action tokens are deeply conditioned on visual observations.")

---
## Step 8: Attention Mask Comparison

Different VLA architectures use different attention masks to control **what can see what**.

The mask determines the information flow — it’s the architectural decision that defines each approach:

1. **OpenVLA**: Full causal mask (like a standard LLM)
2. **pi0**: Block-wise mask (bidirectional within blocks, cross-block rules)
3. **SmolVLA**: Interleaved CA + SA masks (alternating rectangular and triangular)

In [ ]:
# Build all 3 attention masks

# === 1. OpenVLA: Full causal mask ===
# [vision(256) + text(20) + action(7)] = 283 tokens
n_vis_ov, n_txt_ov, n_act_ov = 256, 20, 7
n_total_ov = n_vis_ov + n_txt_ov + n_act_ov  # 283

# Standard causal: everything below and on diagonal is 1
mask_openvla = torch.tril(torch.ones(n_total_ov, n_total_ov))

print(f"1. OpenVLA causal mask: {list(mask_openvla.shape)}")
print(f"   [{n_vis_ov} vision + {n_txt_ov} text + {n_act_ov} action] = {n_total_ov} tokens")
print(f"   Standard lower-triangular causal mask")

# === 2. pi0: Block-wise mask ===
# [vision(256) + text(20) + action(50)] = 326 tokens
n_vis_pi, n_txt_pi, n_act_pi = 256, 20, 50
n_total_pi = n_vis_pi + n_txt_pi + n_act_pi  # 326

mask_pi0 = torch.zeros(n_total_pi, n_total_pi)

# Vision block: bidirectional (full attention within)
mask_pi0[:n_vis_pi, :n_vis_pi] = 1.0

# Text block: causal within text
v = n_vis_pi
t = n_vis_pi + n_txt_pi
for i in range(n_txt_pi):
    mask_pi0[v + i, v:v + i + 1] = 1.0

# Text -> Vision: text tokens can attend to all vision tokens
mask_pi0[v:t, :n_vis_pi] = 1.0

# Action block: causal within action
a = n_vis_pi + n_txt_pi
for i in range(n_act_pi):
    mask_pi0[a + i, a:a + i + 1] = 1.0

# Action -> Vision + Text: action tokens can attend to all vision and text tokens
mask_pi0[a:, :t] = 1.0

# Vision CANNOT see text or actions (blocked)
# (already zero by initialization)

print(f"\n2. pi0 block-wise mask: {list(mask_pi0.shape)}")
print(f"   [{n_vis_pi} vision + {n_txt_pi} text + {n_act_pi} action] = {n_total_pi} tokens")
print(f"   Vision: bidirectional | Text: causal | Action: causal")
print(f"   Cross-block: text->vision OK, action->vision+text OK, vision can't see text/action")

# === 3. SmolVLA: Interleaved CA + SA ===
n_act_sm, n_obs_sm = 50, 64

# CA layer: 50x64 full rectangular (actions query all VLM tokens)
mask_ca = torch.ones(n_act_sm, n_obs_sm)

# SA layer: 50x50 lower triangular (causal self-attention)
mask_sa = torch.tril(torch.ones(n_act_sm, n_act_sm))

print(f"\n3. SmolVLA interleaved CA+SA:")
print(f"   CA mask: {list(mask_ca.shape)} (full rectangular \u2014 actions query all VLM)")
print(f"   SA mask: {list(mask_sa.shape)} (lower triangular \u2014 causal self-attention)")

In [ ]:
# Visualize all 3 attention masks side-by-side
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# --- 1. OpenVLA causal mask (downsampled for visibility) ---
ax = axes[0]
# Downsample to 28x28 for visualization (283 is too large for labels)
ds = 10  # downsample factor
mask_ov_ds = mask_openvla[::ds, ::ds].numpy()
im0 = ax.imshow(mask_ov_ds, cmap='YlOrRd', vmin=0, vmax=1, aspect='equal')

# Mark boundaries
vis_end = n_vis_ov // ds
txt_end = (n_vis_ov + n_txt_ov) // ds
ax.axvline(x=vis_end - 0.5, color=BLUE, linewidth=2, linestyle='--')
ax.axhline(y=vis_end - 0.5, color=BLUE, linewidth=2, linestyle='--')
ax.axvline(x=txt_end - 0.5, color=TEAL, linewidth=2, linestyle='--')
ax.axhline(y=txt_end - 0.5, color=TEAL, linewidth=2, linestyle='--')

# Quadrant labels
ax.text(vis_end / 2, vis_end / 2, 'Vision\n(causal)', ha='center', va='center',
        fontsize=9, color='white', fontweight='bold')
ax.text((vis_end + txt_end) / 2, (vis_end + txt_end) / 2, 'Text\n(causal)', ha='center', va='center',
        fontsize=9, color='white', fontweight='bold')
n_ov_ds = n_total_ov // ds
ax.text((txt_end + n_ov_ds) / 2, (txt_end + n_ov_ds) / 2, 'Act\n(causal)', ha='center', va='center',
        fontsize=9, color='black', fontweight='bold')

ax.set_title('OpenVLA: Full Causal Mask\n[256 vis + 20 txt + 7 act = 283]',
             fontsize=12, color=ACCENT, fontweight='bold')
ax.set_xlabel('Token position (downsampled)', fontsize=10)
ax.set_ylabel('Token position (downsampled)', fontsize=10)

# --- 2. pi0 block-wise mask (downsampled) ---
ax = axes[1]
mask_pi_ds = mask_pi0[::ds, ::ds].numpy()
im1 = ax.imshow(mask_pi_ds, cmap='YlOrRd', vmin=0, vmax=1, aspect='equal')

vis_end_pi = n_vis_pi // ds
txt_end_pi = (n_vis_pi + n_txt_pi) // ds
ax.axvline(x=vis_end_pi - 0.5, color=BLUE, linewidth=2, linestyle='--')
ax.axhline(y=vis_end_pi - 0.5, color=BLUE, linewidth=2, linestyle='--')
ax.axvline(x=txt_end_pi - 0.5, color=TEAL, linewidth=2, linestyle='--')
ax.axhline(y=txt_end_pi - 0.5, color=TEAL, linewidth=2, linestyle='--')

n_pi_ds = n_total_pi // ds
ax.text(vis_end_pi / 2, vis_end_pi / 2, 'Vision\n(bidir)', ha='center', va='center',
        fontsize=9, color='white', fontweight='bold')
ax.text((vis_end_pi + txt_end_pi) / 2, (vis_end_pi + txt_end_pi) / 2, 'Text\n(causal)', ha='center', va='center',
        fontsize=9, color='black', fontweight='bold')
ax.text((txt_end_pi + n_pi_ds) / 2, (txt_end_pi + n_pi_ds) / 2, 'Action\n(causal)', ha='center', va='center',
        fontsize=9, color='black', fontweight='bold')

# Label cross-block regions
ax.text(vis_end_pi / 2, (vis_end_pi + txt_end_pi) / 2, 'text\u2192vis', ha='center', va='center',
        fontsize=8, color='white', fontweight='bold')
ax.text(vis_end_pi / 2, (txt_end_pi + n_pi_ds) / 2, 'act\u2192vis', ha='center', va='center',
        fontsize=8, color='white', fontweight='bold')
ax.text((vis_end_pi + txt_end_pi) / 2, (txt_end_pi + n_pi_ds) / 2, 'act\u2192txt', ha='center', va='center',
        fontsize=8, color='white', fontweight='bold')

ax.set_title('pi0: Block-Wise Mask\n[256 vis + 20 txt + 50 act = 326]',
             fontsize=12, color=TEAL, fontweight='bold')
ax.set_xlabel('Token position (downsampled)', fontsize=10)

# --- 3. SmolVLA interleaved CA+SA ---
ax = axes[2]
# Create a combined visualization: show CA and SA masks stacked/interleaved
# We'll show them as two sub-images in one axes

# Build a composite: pad CA to match SA width and stack vertically
# Top half: CA mask (50x64) padded to 50x64
# Bottom half: SA mask (50x50) padded to 50x64
composite = torch.zeros(n_act_sm * 2 + 5, max(n_obs_sm, n_act_sm))  # with gap
composite[:n_act_sm, :n_obs_sm] = mask_ca
composite[n_act_sm + 5:, :n_act_sm] = mask_sa

im2 = ax.imshow(composite.numpy(), cmap='YlOrRd', vmin=0, vmax=1, aspect='auto')

# Add separation line
ax.axhline(y=n_act_sm + 2, color='black', linewidth=3)

# Labels
ax.text(n_obs_sm / 2, n_act_sm / 2, 'Cross-Attention\n(50\u00d764 full)',
        ha='center', va='center', fontsize=11, color='white', fontweight='bold')
ax.text(n_act_sm / 2, n_act_sm + 5 + n_act_sm / 2, 'Self-Attention\n(50\u00d750 causal)',
        ha='center', va='center', fontsize=11, color='white', fontweight='bold')

ax.set_ylabel('Action tokens (queries)', fontsize=10)
ax.text(0.5, 1.02, 'CA: keys = observations | SA: keys = actions',
        transform=ax.transAxes, ha='center', fontsize=9, color='gray')
ax.set_title('SmolVLA: Interleaved CA + SA\n[50 act \u00d7 64 obs | 50 act \u00d7 50 act]',
             fontsize=12, color=PURPLE, fontweight='bold')

plt.suptitle('Attention Masks: What Can See What?',
             fontsize=16, color=ACCENT, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("OpenVLA (left): Standard causal mask. Every token sees all previous tokens.")
print("  Simple but forces autoregressive generation (slow for actions).")
print("\npi0 (center): Block-wise mask. Vision is bidirectional, text/action are causal.")
print("  Actions can see everything, but vision can't see future text/actions.")
print("  Enables flow matching within the action block.")
print("\nSmolVLA (right): Separate CA and SA masks per block.")
print("  CA: actions fully attend to all observation tokens (rectangular).")
print("  SA: actions causally attend to each other (triangular).")
print("  Cleanest separation of 'read the world' vs 'coordinate actions'.")

---
## Step 9: The Big Comparison + Summary

Let’s bring it all together: how do OpenVLA, pi0, and SmolVLA compare across the key architectural dimensions?

In [ ]:
# Comparison table as matplotlib figure
fig, ax = plt.subplots(figsize=(16, 8))
ax.axis('off')

# Table data
columns = ['Feature', 'OpenVLA', 'pi0', 'SmolVLA']
rows = [
    ['Total Parameters', '7.6B', '3B', '0.4B (VLM) + 0.1B (expert)'],
    ['Visual Encoder', 'SigLIP + DinoV2', 'SigLIP', 'SigLIP'],
    ['Language Backbone', 'Llama-2 7B', 'PaliGemma 3B', 'SmolVLM 0.4B'],
    ['Action Method', 'Autoregressive\n(256-bin tokens)', 'Flow Matching\n(continuous)', 'Flow Matching\n(continuous)'],
    ['Action Expert', 'None (uses LLM)', 'Shared backbone\n(block-wise mask)', 'Separate 100M\n(interleaved CA+SA)'],
    ['Attention Pattern', 'Full causal', 'Block-wise', 'Interleaved CA+SA'],
    ['Chunk Size', '1 step (7 tokens)', '16+ steps', '16+ steps'],
    ['Inference Speed', '~350ms/step\n(7 tokens seq.)', '~100ms/chunk', '~30ms/chunk'],
    ['GPU Required', 'A100 80GB', 'A100 40GB', 'Raspberry Pi 5!'],
]

table = ax.table(
    cellText=rows,
    colLabels=columns,
    loc='center',
    cellLoc='center',
)

# Style the table
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.0, 2.0)

# Header styling
for j in range(len(columns)):
    cell = table[0, j]
    cell.set_facecolor(ACCENT)
    cell.set_text_props(color='white', fontweight='bold', fontsize=12)

# Feature column styling
for i in range(1, len(rows) + 1):
    cell = table[i, 0]
    cell.set_text_props(fontweight='bold')
    cell.set_facecolor('#F0EDE8')

# Alternate row colors
for i in range(1, len(rows) + 1):
    for j in range(1, len(columns)):
        cell = table[i, j]
        if i % 2 == 0:
            cell.set_facecolor('#FAF8F5')
        else:
            cell.set_facecolor('#F5F2ED')

ax.set_title('VLA Architecture Comparison',
             fontsize=16, color=ACCENT, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Parameter count bar chart (log scale)
fig, ax = plt.subplots(figsize=(12, 6))

architectures = ['OpenVLA', 'pi0', 'SmolVLA\n(VLM + Expert)']
params = [7.6e9, 3.0e9, 0.5e9]  # total params
colors = [ACCENT, TEAL, PURPLE]

bars = ax.bar(architectures, params, color=colors, edgecolor='white', linewidth=2, width=0.6)

ax.set_yscale('log')
ax.set_ylabel('Total Parameters (log scale)', fontsize=12, fontweight='bold')
ax.set_title('VLA Parameter Counts', fontsize=14, color=ACCENT, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Add labels on bars
for bar, p, c in zip(bars, params, colors):
    label = f'{p/1e9:.1f}B'
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.5,
            label, ha='center', fontsize=14, fontweight='bold', color=c)

# Add speedup annotation
ax.annotate(f'{7.6/0.5:.0f}x fewer\nparams!',
            xy=(2, 0.5e9), xytext=(1.5, 2e9),
            fontsize=13, color=PURPLE, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=PURPLE, lw=2))

ax.set_ylim(1e8, 2e10)
plt.tight_layout()
plt.show()

print("The trend is clear: newer VLAs achieve better performance with FEWER parameters.")
print("Key innovations enabling this:")
print("  1. Flow matching instead of autoregressive (parallel action generation)")
print("  2. Separate action expert instead of using the full LLM")
print("  3. Cross-attention for conditioning instead of concatenating everything")

---
## Summary

| Concept | What We Built | Key Takeaway |
|---------|--------------|-------------|
| Self-Attention | `SelfAttention` in 8 lines | Square N\u00d7N matrix — tokens attend to themselves |
| Action Tokenization | 256-bin quantile encode/decode | Continuous \u2192 discrete \u2192 continuous with ~0.001 error |
| Autoregressive Bottleneck | Timing simulation | 7B model \u00d7 112 tokens = 5.6s per chunk |
| Cross-Attention | `CrossAttention` module | **Only difference: K,V from different source** \u2192 rectangular matrix |
| Conditioning | Reaching\u2192cup, placing\u2192bowl demo | Cross-attention IS the conditioning mechanism |
| OpenVLA Pipeline | Dual encoder + MLP + autoregressive | 7.6B params, 350ms/step |
| SmolVLA Expert | 6 interleaved CA+SA blocks | 100M params, 30ms/chunk |
| Attention Masks | 3 mask visualizations | The mask defines the architecture |

### The Evolution of VLA Architectures

1. **OpenVLA** (2024): Take a powerful VLM, fine-tune it to output discrete action tokens. Simple but slow (autoregressive) and huge (7B).

2. **pi0** (2024): Use flow matching instead of autoregressive generation. Block-wise attention mask lets action tokens attend to everything while generating in parallel.

3. **SmolVLA** (2025): Separate the action expert from the VLM backbone entirely. Tiny cross-attention expert (100M) reads from a small VLM (400M). Runs on a Raspberry Pi.

The key insight: **cross-attention decouples understanding from acting**. The VLM understands the scene (once), and the action expert queries it (many times during denoising).

---

## Exercises

1. **Tokenization precision:** Change `N_BINS` to 64, 128, 256, and 512. Compute the reconstruction MSE for each. Plot MSE vs number of bins. At what point do diminishing returns kick in?

2. **CA-only vs SA-only ablation:** Build two variants of `SmolVLAActionExpertBlock` — one with only cross-attention (remove SA), one with only self-attention (remove CA). Generate a 50-step trajectory from random noise using each. Compute trajectory smoothness as the mean autocorrelation of consecutive action differences: `autocorr = mean(cosine_sim(delta_t, delta_{t+1}))`. Which variant produces smoother trajectories?

3. **Number of interleaved blocks:** Build SmolVLA action experts with 2, 4, 6, and 8 blocks. Compare: (a) total parameter count, (b) L2 distance between 2-block and N-block outputs given the same input. Plot both. How many blocks are needed before outputs converge?

4. **Causal vs bidirectional SA:** Modify `SelfAttention` to apply a causal mask (add `-1e9` to the upper triangle of scores before softmax). Compare the attention maps of causal vs bidirectional SA on the same 50 action tokens. How does the output change? When would you prefer one over the other?

5. **Challenge: Build a mini-VLA.** Create a complete mini-VLA that:
   - Takes a 3-channel 32\u00d732 image (use a small CNN to get 16 patch tokens of dim 64)
   - Takes a text command (embed 5 text tokens of dim 64)
   - Uses 3 `SmolVLAActionExpertBlock` layers to generate 7-dim continuous actions
   - Train it on a synthetic dataset: image has a colored dot at position (x, y), text says "go to dot", target action is (x, y, 0, 0, 0, 0, 1). Does cross-attention learn to attend to the dot’s patch?